# Lightweight HBCC — Food-101 Training Pipeline

Trains all models on ETHZ Food-101 (101 classes, 75 750 train / 25 250 test).
Images are resized to 32×32 to match the HBCC CIFAR configuration exactly.

**Required Kaggle dataset:** `dansbecker/food-101`  
Add it via *Add Data → Datasets* before running. On Kaggle's filesystem it mounts at:
```
/kaggle/input/datasets/dansbecker/food-101/
  food-101/
    images/<class_name>/<img_id>.jpg
    meta/train.txt
    meta/test.txt
```

**Validation split:** 7 575 images (10 %) are carved from `train.txt` with seed=42.  
The official Food-101 test split (25 250 images, manually reviewed by ETH Zurich) is the test set.

**Two sessions required** (100 epochs each; Food-101 has ≈1.5× more batches/epoch than CIFAR-10):

| Session | What runs | Est. T4 time |
|---------|-----------|-------------|
| **A** | ResNet-18 + MobileNetV2 + ShuffleNetV2 + CoC | ~5.4 h |
| **B** | HBCC-Small CE → HBCC-Medium CE → Teacher re-train → HBCC-Small KD → HBCC-Medium KD → benchmarks | ~10.6 h |

Enable **only** the phases for the current session before committing.

In [ ]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster
!pip install fvcore -q

In [ ]:
# ── Extract Food-101 dataset ───────────────────────────────────────────────────
# dansbecker/food-101 is a multi-level nested zip (macOS-created, ZIP64):
#   /kaggle/input/.../food-101.zip  →  food-101.zip  →  food-101/images/...
#
# unzip 6.0 (Kaggle Ubuntu) rejects ZIP64 files whose local-header offset is
# stored in the ZIP64 extra field — it misreads the 8-byte value as
# "4294967296 extra bytes" and raises a zip-bomb error.
# 7z (p7zip-full) handles all ZIP64 variants correctly and is used instead.
import shutil as _shutil
import subprocess
import time as _time
import zipfile
from pathlib import Path as _Path

_food101_zip   = _Path("/kaggle/input/datasets/dansbecker/food-101/food-101.zip")
_food101_check = _Path("/kaggle/working/food-101/images")
_wk            = _Path("/kaggle/working")


def _ensure_7z() -> None:
    if _shutil.which("7z"):
        return
    print("Installing p7zip-full…")
    r = subprocess.run(
        ["apt-get", "install", "-y", "-q", "p7zip-full"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"apt-get install p7zip-full failed:\n{r.stderr}")
    print("p7zip-full installed.")


def _unzip(src: _Path, dest: _Path) -> None:
    """Extract zip via 7z (handles ZIP64 that unzip 6.0 rejects as a zip bomb)."""
    _ensure_7z()
    # -o must be immediately followed by the path (no space) in 7z syntax.
    r = subprocess.run(
        ["7z", "x", str(src), f"-o{dest}", "-y"],
        capture_output=True, text=True,
    )
    if r.returncode > 1:  # 0 = ok, 1 = warning-only
        raise RuntimeError(f"7z failed (code {r.returncode}):\n{r.stderr or r.stdout}")


def _fix_structure() -> bool:
    """Ensure food-101/images/ exists at the expected path. Returns True when done."""
    if _food101_check.exists():
        return True
    # Flat: images/ and meta/ landed directly in /kaggle/working/
    if (_wk / "images").is_dir() and (_wk / "meta").is_dir():
        d = _wk / "food-101"
        d.mkdir(exist_ok=True)
        _shutil.move(str(_wk / "images"), str(d / "images"))
        _shutil.move(str(_wk / "meta"),   str(d / "meta"))
        print("Restructured flat layout → food-101/")
    else:
        # Different top-level directory name (Food-101/, food101/, etc.)
        _cands = [p for p in _wk.iterdir() if p.is_dir() and (p / "images").is_dir()]
        if _cands and _cands[0].name != "food-101":
            _cands[0].rename(_wk / "food-101")
            print(f"Renamed '{_cands[0].name}/' → 'food-101/'")
    return _food101_check.exists()


if _food101_check.exists():
    print(f"food-101 already extracted: {_food101_check}")
else:
    if not _food101_zip.exists():
        raise FileNotFoundError(
            f"food-101.zip not found: {_food101_zip}\n"
            "Add 'dansbecker/food-101' as a Kaggle dataset input."
        )

    _t0 = _time.perf_counter()

    # Step 1: extract the Kaggle-mounted outer zip
    with zipfile.ZipFile(_food101_zip) as _z:
        _outer_top = sorted({n.split("/")[0] for n in _z.namelist() if n.strip("/")})
    print(f"Outer zip top-level: {_outer_top[:8]}")
    _unzip(_food101_zip, _wk)

    # Step 2: keep extracting nested food-101.zip files until images/ appears.
    # Each iteration renames the inner zip before extracting to avoid
    # source/destination collision when the zip and its contents share a name.
    for _depth in range(1, 6):
        if _fix_structure():
            break

        _inner = _wk / "food-101.zip"
        if not _inner.exists():
            raise FileNotFoundError(
                f"food-101/images not found and no inner zip at depth {_depth}.\n"
                f"Contents of /kaggle/working: {[p.name for p in sorted(_wk.iterdir())]}"
            )

        _tmp = _wk / f"_food101_level{_depth}.zip"
        _inner.rename(_tmp)
        with zipfile.ZipFile(_tmp) as _z:
            _inner_top = sorted({n.split("/")[0] for n in _z.namelist() if n.strip("/")})
        print(f"[level {_depth}] inner zip top-level: {_inner_top[:8]}")
        _unzip(_tmp, _wk)
        _tmp.unlink()  # free ~5 GB of intermediate zip
    else:
        raise FileNotFoundError("Gave up after 5 extraction levels — food-101/images not found.")

    print(f"Extraction complete in {_time.perf_counter() - _t0:.0f}s")
    print(f"Ready: {_food101_check}")
    print(f"Meta : {_wk / 'food-101' / 'meta'}")

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Repo root not found: {ROOT}"

PYTHON = Path(sys.executable)
print("Repo  :", ROOT)
print("Python:", PYTHON)

## Phase Switches

**Session A:** set `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.  
**Session B:** set `RUN_STUDENT_CE`, `RUN_TEACHER_KD`, `RUN_KD`, `RUN_BENCHMARKS` = True; all others False.

> **Cross-session accuracy:** Session A saves a partial metrics CSV to `results/`.  
> In Session B, fill in `PREV_SESSION_A_METRICS` with the path to that CSV so the  
> final publication table includes all 8 models.
>
> To find the CSV path: open the committed Session A run → Output tab → hover over  
> `results/food101_session_a.csv` → copy path.

In [ ]:
# ── Session A ──────────────────────────────────────────────────────────────────
RUN_ENV_CHECKS            = True   # run only in Session A
RUN_TEACHER               = True   # ResNet-18 reference baseline
RUN_LIGHTWEIGHT_BASELINES = True   # MobileNetV2, ShuffleNetV2
RUN_COC_BASELINE          = True   # CoC pure-cluster baseline

# ── Session B ──────────────────────────────────────────────────────────────────
RUN_STUDENT_CE            = False  # HBCC-Small CE + HBCC-Medium CE  (~4.7 h)
RUN_TEACHER_KD            = False  # re-train ResNet-18 as KD teacher (~1.3 h)
RUN_KD                    = False  # HBCC-Small KD + HBCC-Medium KD  (~4.7 h)
RUN_BENCHMARKS            = False  # latency / params / MACs          (~0.5 h)

# ── Epoch budgets ───────────────────────────────────────────────────────────────
# Food-101: ~68 k train images ≈ 1.5× CIFAR-10 batches/epoch at batch_size=128.
# 100 epochs keeps both sessions safely within Kaggle's 12 h GPU limit.
# Timing estimates on T4 (conservative):
#   Session A: env(~0.3h) + Teacher(~1.3h) + MobileNetV2(~1.3h)
#              + ShuffleNetV2(~1.0h) + CoC(~1.5h) ≈ 5.4 h
#   Session B: HBCC-Small CE(~2.2h) + HBCC-Medium CE(~2.4h) + Teacher KD(~1.3h)
#              + HBCC-Small KD(~2.2h) + HBCC-Medium KD(~2.4h) + bench(~0.5h) ≈ 11.0 h
BASELINE_EPOCHS = 100   # ResNet-18, MobileNetV2, ShuffleNetV2, CoC
HBCC_CE_EPOCHS  = 100   # HBCC-Small and HBCC-Medium CE
KD_EPOCHS       = 100   # HBCC-Small and HBCC-Medium KD

# ── Reproducible seed ──────────────────────────────────────────────────────────
SEED = 42
COMMON_TRAIN_OVERRIDES = [f"train.seed={SEED}"]

# ── Dataset ────────────────────────────────────────────────────────────────────
# food-101.zip is extracted to /kaggle/working/ by the cell above.
# torchvision.datasets.Food101 with root="/kaggle/working" then reads from:
#   /kaggle/working/food-101/images/<class>/<img>.jpg
#   /kaggle/working/food-101/meta/train.txt  (75 750 entries)
#   /kaggle/working/food-101/meta/test.txt   (25 250 entries)
_FOOD101_ROOT = "/kaggle/working"

FOOD101_BASE = [
    "data.name=food101",
    f"data.root={_FOOD101_ROOT}",
    "data.download=false",
    "model.num_classes=101",
    "data.workers=4",
]

# ── Benchmark settings ─────────────────────────────────────────────────────────
BENCHMARK_OUTPUT        = "results/benchmark_food101"
BENCHMARK_BATCHES       = [1, 16, 64, 128]
BENCHMARK_WARMUP        = 30
BENCHMARK_RUNS          = 100
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP  = 2
DEBUG_BENCHMARK_RUNS    = 3

# ── Cross-session accuracy (Session B only) ────────────────────────────────────
# After committing Session A: download results/food101_session_a.csv, then paste its path here.
PREV_SESSION_A_METRICS = ""  # e.g. "/kaggle/input/hbcc-f101-a/results/food101_session_a.csv"

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def f101_overrides(
    experiment_name: str | None = None,
    extra_overrides: list[str] | None = None,
) -> list[str]:
    """Build the full override list for a Food-101 run."""
    items = [*FOOD101_BASE]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(extra_overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    all_overrides = f101_overrides(experiment_name, [*(overrides or []), *COMMON_TRAIN_OVERRIDES])
    args += override_args(all_overrides)
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup  = DEBUG_BENCHMARK_WARMUP  if debug else BENCHMARK_WARMUP
    runs    = DEBUG_BENCHMARK_RUNS    if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py", "--config", config,
        "--output", BENCHMARK_OUTPUT,
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    # model.num_classes=101 ensures the benchmark measures the correct classifier head.
    args += override_args(["model.num_classes=101", f"experiment.name={experiment_name}"])
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"[warn] Checkpoint not found — benchmarking architecture only: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def save_session_metrics(session_label: str, run_dirs: list[str]) -> Path:
    """Scan run_dirs for metrics.jsonl files and save a per-session CSV."""
    rows = []
    for run_root in run_dirs:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })
    out_dir = ROOT / "results"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{session_label}.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"Saved {len(rows)} rows → {out_path}")
    return out_path

## 0. Environment Checks

Run only in **Session A** (`RUN_ENV_CHECKS = True`).  
Verifies the Food-101 dataset path, class count, and entry counts, then runs  
the unit-test suite and traces HBCC-Medium shapes for 101 classes.

In [ ]:
if RUN_ENV_CHECKS:
    food101_images    = Path(_FOOD101_ROOT) / "food-101" / "images"
    food101_train_txt = Path(_FOOD101_ROOT) / "food-101" / "meta" / "train.txt"
    food101_test_txt  = Path(_FOOD101_ROOT) / "food-101" / "meta" / "test.txt"

    if not food101_images.exists():
        raise FileNotFoundError(
            f"Food-101 images dir not found: {food101_images}\n"
            "Add 'dansbecker/food-101' as a Kaggle dataset input and re-run."
        )
    if not food101_train_txt.exists():
        raise FileNotFoundError(f"Food-101 meta/train.txt not found: {food101_train_txt}")

    classes     = sorted(d.name for d in food101_images.iterdir() if d.is_dir())
    train_lines = [l for l in food101_train_txt.read_text().splitlines() if l.strip()]
    test_lines  = [l for l in food101_test_txt.read_text().splitlines()  if l.strip()] \
                  if food101_test_txt.exists() else []

    print(f"Classes   : {len(classes)}")
    print(f"Train IDs : {len(train_lines)}")
    print(f"Test IDs  : {len(test_lines)}")

    assert len(classes) == 101,       f"Expected 101 classes, found {len(classes)}"
    assert len(train_lines) == 75750, f"Expected 75 750 train entries, got {len(train_lines)}"
    assert len(test_lines)  == 25250, f"Expected 25 250 test entries, got {len(test_lines)}"
    print("OK: Food-101 dataset verified (101 classes, 75 750 train, 25 250 test)")

    run_py(["-m", "pytest", "-q"])

    run_py([
        "tools/shape_trace.py", "--config", "configs/hbcc_accuracy_medium.yaml",
        "--batch-size", "1",
        *override_args(["model.num_classes=101"]),
    ])

## Session A — Teacher and Lightweight Baselines

All four models trained for `BASELINE_EPOCHS` (100) with **full augmentation**.

Estimated time on Kaggle T4: **~5.4 h** (well within the 12 h limit).

After committing: download `results/food101_session_a.csv` from the Output tab, then  
paste its path into `PREV_SESSION_A_METRICS` before running Session B.

In [ ]:
EXP_RESNET  = "resnet18_f101"
DIR_TEACHER = "runs_f101_teacher"
TEACHER_CHECKPOINT = f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"

if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER,
        EXP_RESNET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
EXP_MOBILENET  = "mobilenet_v2_f101"
EXP_SHUFFLENET = "shufflenet_v2_x1_0_f101"
DIR_BASELINES  = "runs_f101_baselines"

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        DIR_BASELINES,
        EXP_MOBILENET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        DIR_BASELINES,
        EXP_SHUFFLENET,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
EXP_COC = "coc_baseline_f101"
DIR_COC = "runs_f101_coc"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        DIR_COC,
        EXP_COC,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
# Save Session A metrics. Download food101_session_a.csv after committing;
# paste its path into PREV_SESSION_A_METRICS before running Session B.
if any([RUN_TEACHER, RUN_LIGHTWEIGHT_BASELINES, RUN_COC_BASELINE]):
    save_session_metrics("food101_session_a", [DIR_TEACHER, DIR_BASELINES, DIR_COC])

## Session B — HBCC CE, KD, Benchmarks, and Publication Table

Runs in this order to stay within 12 h:

1. **HBCC-Small CE** (~2.2 h) — no teacher needed, starts immediately
2. **HBCC-Medium CE** (~2.4 h) — no teacher needed
3. **ResNet-18 KD teacher re-train** (~1.3 h) — must finish before KD steps
4. **HBCC-Small KD** (~2.2 h)
5. **HBCC-Medium KD** (~2.4 h)
6. **Benchmarks** (~0.5 h)

Estimated total on Kaggle T4: **~11.0 h** (within the 12 h limit).

> **Before running:** fill in `PREV_SESSION_A_METRICS` in the phase-switches cell  
> so the final publication table includes all 8 models.
>
> **To skip the ~1.3 h teacher re-train** and reduce Session B to ~9.7 h, set  
> `RUN_TEACHER_KD = False` and point `TEACHER_KD_CHECKPOINT` to the Session A  
> checkpoint you added as a dataset input:
> ```python
> TEACHER_KD_CHECKPOINT = "/kaggle/input/<your-dataset>/runs_f101_teacher/resnet18_f101/best.pth"
> ```

In [ ]:
EXP_HBCC_SML = "hbcc_accuracy_small_f101"
EXP_HBCC_MED = "hbcc_accuracy_medium_f101"
DIR_ACCURACY = "runs_f101_ce"

if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_ACCURACY,
        EXP_HBCC_SML,
        overrides=[f"train.epochs={HBCC_CE_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_ACCURACY,
        EXP_HBCC_MED,
        overrides=[f"train.epochs={HBCC_CE_EPOCHS}"],
    )

In [ ]:
EXP_RESNET_KD         = "resnet18_f101_kd"
DIR_TEACHER_KD        = "runs_f101_teacher_kd"
TEACHER_KD_CHECKPOINT = f"{DIR_TEACHER_KD}/{EXP_RESNET_KD}/best.pth"

# Override with Session A checkpoint if you added it as a dataset input (saves ~1.3 h):
# TEACHER_KD_CHECKPOINT = "/kaggle/input/<your-dataset>/runs_f101_teacher/resnet18_f101/best.pth"

if RUN_TEACHER_KD:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER_KD,
        EXP_RESNET_KD,
        overrides=[f"train.epochs={BASELINE_EPOCHS}"],
    )

In [ ]:
EXP_HBCC_SML_KD = "hbcc_accuracy_small_kd_f101"
EXP_HBCC_MED_KD = "hbcc_accuracy_medium_kd_f101"
DIR_KD          = "runs_f101_kd"

KD_OVERRIDES = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

if RUN_KD:
    teacher_path = ROOT / TEACHER_KD_CHECKPOINT
    if not teacher_path.exists():
        raise FileNotFoundError(
            f"Teacher checkpoint not found: {teacher_path}\n"
            "Set RUN_TEACHER_KD=True, or point TEACHER_KD_CHECKPOINT to an existing checkpoint."
        )
    print(f"Teacher checkpoint: {teacher_path}")

    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_KD,
        EXP_HBCC_SML_KD,
        overrides=[f"train.epochs={KD_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config",     "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_KD,
        EXP_HBCC_MED_KD,
        overrides=[f"train.epochs={KD_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config",     "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )

In [ ]:
# Save Session B metrics (HBCC CE + KD variants).
if any([RUN_STUDENT_CE, RUN_TEACHER_KD, RUN_KD]):
    save_session_metrics("food101_session_b", [DIR_ACCURACY, DIR_TEACHER_KD, DIR_KD])

## Benchmark Matrix

Measures latency, throughput, params, and peak memory for all models.  
Baselines (Teacher, MobileNetV2, ShuffleNetV2, CoC) are benchmarked with random weights  
since their checkpoints are from Session A — this is correct because latency and memory  
are architecture properties, not training outcomes.  
The teacher (re-trained this session) and KD checkpoints load real weights.

In [ ]:
# (config, experiment_name, checkpoint_or_None)
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",            EXP_RESNET,       f"{DIR_TEACHER_KD}/{EXP_RESNET_KD}/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",        EXP_MOBILENET,    None),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", EXP_SHUFFLENET,   None),
    ("configs/coc_cifar_baseline.yaml",                  EXP_COC,          None),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML,     None),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED,     None),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML_KD,  f"{DIR_KD}/{EXP_HBCC_SML_KD}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED_KD,  f"{DIR_KD}/{EXP_HBCC_MED_KD}/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print(f"Skip missing config: {cfg}")
            continue
        benchmark(cfg, exp_name, checkpoint=ckpt)

## Results Tables

These cells can be run in any session and show whatever data is currently available.  
For the **complete publication table** you need Session B to have run with  
`PREV_SESSION_A_METRICS` filled in.

In [ ]:
F101_RUN_DIRS = [
    DIR_TEACHER, DIR_BASELINES, DIR_COC,   # Session A dirs (present only in Session A)
    DIR_ACCURACY, DIR_TEACHER_KD, DIR_KD,  # Session B dirs
]


def collect_train_metrics() -> pd.DataFrame:
    """Collect from this session's run dirs, then merge the Session A CSV if provided."""
    rows = []
    for run_root in F101_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })

    _cols = ["experiment", "best_val_acc1", "best_val_acc5", "test_acc1", "test_acc5"]
    df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)

    if PREV_SESSION_A_METRICS:
        p = Path(PREV_SESSION_A_METRICS)
        if p.exists():
            prev_df  = pd.read_csv(p)
            existing = set(df["experiment"])
            new_rows = prev_df[~prev_df["experiment"].isin(existing)]
            df = pd.concat([df, new_rows], ignore_index=True)
            print(f"Merged {len(new_rows)} rows from {p.name}")
        else:
            print(f"[warn] PREV_SESSION_A_METRICS not found (skipped): {p}")

    if df.empty:
        return df
    return df.sort_values("best_val_acc1", ascending=False).reset_index(drop=True)


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        try:
            rows.append(json.loads(path.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            print(f"[warn] Could not parse benchmark JSON: {path.name}")
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
arch_cols = ["config_id", "model_name", "params_total", "macs", "latency_ms_b1", "peak_memory_mb"]
bench_df[[c for c in arch_cols if c in bench_df.columns]] if not bench_df.empty else bench_df

In [ ]:
def build_publication_table(bench: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        print("No benchmark data yet — run Session B with RUN_BENCHMARKS=True.")
        return bench
    out = bench.copy()
    out["acc1"] = None
    out["acc5"] = None
    if not train.empty:
        acc1_key = "test_acc1" if "test_acc1" in train.columns else "best_val_acc1"
        acc5_key = "test_acc5" if "test_acc5" in train.columns else "best_val_acc5"
        acc1_map = dict(zip(train["experiment"], train[acc1_key].fillna(train["best_val_acc1"])))
        acc5_map = dict(zip(train["experiment"], train[acc5_key])) if acc5_key in train.columns else {}
        for idx, row in out.iterrows():
            name = row.get("config_id")
            if name in acc1_map:
                out.at[idx, "acc1"] = acc1_map[name]
            if name in acc5_map:
                out.at[idx, "acc5"] = acc5_map[name]
    rename = {
        "config_id":      "Model",
        "acc1":           "Top-1 Acc (%)",
        "acc5":           "Top-5 Acc (%)",
        "params_total":   "Params",
        "macs":           "MACs",
        "latency_ms_b1":  "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    table = out[[c for c in rename if c in out.columns]].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.dropna(subset=["Top-1 Acc (%)"])
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


pub_df = build_publication_table(bench_df, train_df)
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
pub_df.to_csv(out_dir / "food101_published_results.csv", index=False)
pub_df

## Recommended Execution Order

**Session A (~5.4 h):**
1. Set `RUN_ENV_CHECKS`, `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.
2. Run all cells and commit.
3. Open the Output tab → download `results/food101_session_a.csv`.

**Session B (~11.0 h):**
1. Set `RUN_STUDENT_CE`, `RUN_TEACHER_KD`, `RUN_KD`, `RUN_BENCHMARKS` = True; all others False.
2. Paste the Session A CSV path into `PREV_SESSION_A_METRICS`.
3. *(Optional)* To save ~1.3 h: add the committed Session A output as a Kaggle dataset input,  
   set `RUN_TEACHER_KD = False`, and point `TEACHER_KD_CHECKPOINT` to the Session A checkpoint.
4. Run all cells and commit.

The publication table and benchmark CSV are written to `results/` at the end of Session B.